# **IMDB Word2Vec: Pipeline 3**

This experiment implements a classical NLP baseline using the
**Word2Vec** representation over the raw IMDB dataset
without preprocessing transformations.

Pipeline 2 performs the following operations sequentially:

1. Remove special characters and numbers
2. Tokenize the text into words
3. Convert all tokens to lowercase
4. StopWord Removal
5. Reconstruct the cleaned text

The objective of this experiment is to:

- Build Bag of Words representations
- Train baseline machine learning classifiers
- Evaluate classification performance
- Compare traditional NLP approaches against future embedding architectures



# **1. Import Libraries**

In [2]:
import pandas as pd
import numpy as np
import os
import time

from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import LinearSVC

In [3]:
#import sys
#!{sys.executable} -m pip install gensim

# **2. Load Dataset**

Pipeline 0 uses the raw IMDB dataset without preprocessing.

The dataset is divided into:
- Training set
- Validation set
- Testing set

In [4]:
train = pd.read_csv(
    r"C:\Users\Ale\Documents\Final_NLP\Homework_2\Text-Representation-and-Basic-Text-Mining\Text_Classification_and_Embeddings\Models\Data\preprocessed_data\pipeline_3\train.csv"
)

val = pd.read_csv(
    r"C:\Users\Ale\Documents\Final_NLP\Homework_2\Text-Representation-and-Basic-Text-Mining\Text_Classification_and_Embeddings\Models\Data\preprocessed_data\pipeline_3\validation.csv"
)

test = pd.read_csv(
    r"C:\Users\Ale\Documents\Final_NLP\Homework_2\Text-Representation-and-Basic-Text-Mining\Text_Classification_and_Embeddings\Models\Data\preprocessed_data\pipeline_3\test.csv"
)

print("Train Shape:", train.shape)
print("Validation Shape:", val.shape)
print("Test Shape:", test.shape)

Train Shape: (20000, 2)
Validation Shape: (2500, 2)
Test Shape: (2500, 2)


# **3. Word Embeddings**


In [5]:
def compute_word2vec(
    X_train,
    X_val,
    X_test,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4
):

    """
    Computes Word2Vec sentence embeddings
    for train, validation, and test sets.

    Steps:
    1. Tokenize text
    2. Train Word2Vec model
    3. Generate sentence embeddings
       using average word vectors

    Returns:
    - X_train_w2v
    - X_val_w2v
    - X_test_w2v
    - trained Word2Vec model
    """

    # -----------------------------------------
    # Tokenization
    # -----------------------------------------

    train_tokens = [
        word_tokenize(text.lower())
        for text in X_train
    ]

    val_tokens = [
        word_tokenize(text.lower())
        for text in X_val
    ]

    test_tokens = [
        word_tokenize(text.lower())
        for text in X_test
    ]

    # Train Word2Vec Model
    word2vec_model = Word2Vec(
        sentences=train_tokens,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers
    )

    # Sentence Embedding Function
    def average_word_vectors(tokens):
        vectors = []
        for word in tokens:
            if word in word2vec_model.wv:
                vectors.append(
                    word2vec_model.wv[word]
                )

        # If sentence has no valid words
        if len(vectors) == 0:
            return np.zeros(vector_size)

        # Average embeddings
        return np.mean(vectors, axis=0)

    # Generate Sentence Embeddings
    X_train_w2v = np.array([
        average_word_vectors(tokens)
        for tokens in train_tokens
    ])

    X_val_w2v = np.array([
        average_word_vectors(tokens)
        for tokens in val_tokens
    ])

    X_test_w2v = np.array([
        average_word_vectors(tokens)
        for tokens in test_tokens
    ])

    return (
        X_train_w2v,
        X_val_w2v,
        X_test_w2v,
        word2vec_model
    )

# **4. Prepare Features and Labels**

In [6]:
X_train = train['text']
y_train = train['label']

X_val = val['text']
y_val = val['label']

X_test = test['text']
y_test = test['label']

In [7]:
X_train_w2v, X_val_w2v, X_test_w2v, word2vec_model = compute_word2vec(
    X_train,
    X_val,
    X_test
)

print("Word2Vec Shapes:")
print("Train:", X_train_w2v.shape)
print("Validation:", X_val_w2v.shape)
print("Test:", X_test_w2v.shape)

Word2Vec Shapes:
Train: (20000, 300)
Validation: (2500, 300)
Test: (2500, 300)


# **5. Model Training**

Two classical machine learning classifiers are trained:

- Logistic Regression
- Multinomial Naive Bayes

These models are commonly used as traditional NLP baselines.

In [8]:
def train_model(model, X_train, y_train):

    """
    Trains a machine learning model.
    """
    model.fit(X_train, y_train)
    return model

In [9]:
# Logistic Regression
logistic_model = LogisticRegression(max_iter=1000)

logistic_model = train_model(
    logistic_model,
    X_train_w2v,
    y_train
)

In [10]:
# SuperVector Machine
svm_model = LinearSVC()

svm_model = train_model(
    svm_model,
    X_train_w2v,
    y_train
)

# **6. Evaluation Metrics**

The following metrics are computed:

- Accuracy
- Precision
- Recall
- F1-Score

These metrics evaluate classification performance
from different perspectives.

In [11]:

def compute_metrics(model, X_test, y_test):

    """
    Computes classification metrics
    and inference time.
    """

    start_time = time.time()
    # Predictions
    predictions = model.predict(X_test)
    end_time = time.time()
    inference_time = end_time - start_time

    # Average inference time per sample
    avg_inference_time = inference_time / len(y_test)

    # Compute metrics
    metrics = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1-Score": f1_score(y_test, predictions),
        "Inference_Time_Seconds": inference_time,
        "Average_Inference_Time_Per_Sample":avg_inference_time      
    }

    return metrics, predictions

In [12]:
# Logistic Regression Metrics
log_metrics, log_predictions = compute_metrics(
    logistic_model,
    X_test_w2v,
    y_test
)

print("Logistic Regression Metrics")
print(log_metrics)

Logistic Regression Metrics
{'Accuracy': 0.8492, 'Precision': 0.8381099922540666, 'Recall': 0.8656, 'F1-Score': 0.8516332152695789, 'Inference_Time_Seconds': 0.0020012855529785156, 'Average_Inference_Time_Per_Sample': 8.005142211914063e-07}


In [13]:
# SVM Metrics
svm_metrics, svm_predictions = compute_metrics(
    svm_model,
    X_test_w2v,
    y_test
)

print("\nSVM Metrics")
print(svm_metrics)


SVM Metrics
{'Accuracy': 0.858, 'Precision': 0.8460943542150039, 'Recall': 0.8752, 'F1-Score': 0.8604011010617381, 'Inference_Time_Seconds': 0.0020046234130859375, 'Average_Inference_Time_Per_Sample': 8.01849365234375e-07}


# **7. Confusion Matrix**

The confusion matrix allows visualization of:

- True Positives
- True Negatives
- False Positives
- False Negatives

This helps analyze classification behavior and prediction errors.

In [14]:
def plot_confusion_matrix(
    y_true,
    y_pred,
    model_name,
    pipeline_name,
    representation_name,
    output_dir="results_w2v"
):

    """
    Computes and saves confusion matrix.

    The confusion matrix includes:
    - Model name
    - Pipeline name
    - Representation type

    Confusion matrices are saved inside:
    conf_matrix/
    """

    conf_matrix_dir = os.path.join(
        output_dir,
        "conf_matrix"
    )

    os.makedirs(conf_matrix_dir, exist_ok=True)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues'
    )

    plt.title(
        f"{model_name} | {pipeline_name} | {representation_name}"
    )

    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    file_name = (
        f"{model_name}_"
        f"{pipeline_name}_"
        f"{representation_name}_"
        f"confusion_matrix.png"
    )

    save_path = os.path.join(
        conf_matrix_dir,
        file_name
    )

    plt.savefig(
        save_path,
        bbox_inches='tight'
    )

    plt.close()

    print(f"Confusion matrix saved in:\n{save_path}")

In [15]:
# Create Results Directory
results_dir = "results_w2v"
os.makedirs(results_dir, exist_ok=True)


# Plot Confusion Matrices
plot_confusion_matrix(
    y_true=y_test,
    y_pred=log_predictions,
    model_name="LogisticRegression",
    pipeline_name="pipeline_3",
    representation_name="Word2Vec"
)

plot_confusion_matrix(
    y_true=y_test,
    y_pred=svm_predictions,
    model_name="SVM",
    pipeline_name="pipeline_3",
    representation_name="Word2Vec"
)

Confusion matrix saved in:
results_w2v\conf_matrix\LogisticRegression_pipeline_3_Word2Vec_confusion_matrix.png
Confusion matrix saved in:
results_w2v\conf_matrix\SVM_pipeline_3_Word2Vec_confusion_matrix.png


# **8. Save Results**

All evaluation metrics and reports are saved
inside the `results_bag/` directory for future analysis.

In [16]:
def save_results(
    metrics,
    model_name,
    pipeline_name,
    representation_name,
    output_dir="results_w2v",
    results_file="experiment_results.csv"
):

    """
    Saves experiment results into a single CSV file.
    Each experiment is appended as a new row.
    """

    # Create results directory
    os.makedirs(output_dir, exist_ok=True)

    # Full CSV path
    save_path = os.path.join(
        output_dir,
        results_file
    )

    # Create experiment dictionary
    results = {
        "Model": model_name,
        "Pipeline": pipeline_name,
        "Representation": representation_name,
        **metrics
    }

    # Convert current experiment to dataframe
    current_result_df = pd.DataFrame([results])

    # Append if file exists
    if os.path.exists(save_path):
        existing_results = pd.read_csv(save_path)
        updated_results = pd.concat(
            [existing_results, current_result_df],
            ignore_index=True
        )
    else:
        updated_results = current_result_df

    # Save updated dataframe
    updated_results.to_csv(
        save_path,
        index=False
    )

    print(f"Results appended to:\n{save_path}")

In [17]:
# Save Logistic Regression Results
save_results(
    metrics=log_metrics,
    model_name="LogisticRegression",
    pipeline_name="pipeline_3",
    representation_name="Word2Vec"
)

# Save SVM Results
save_results(
    metrics=svm_metrics,
    model_name="SVM",
    pipeline_name="pipeline_3",
    representation_name="Word2Vec"
)
    

print("Results saved successfully inside:")
print(results_dir)

Results appended to:
results_w2v\experiment_results.csv
Results appended to:
results_w2v\experiment_results.csv
Results saved successfully inside:
results_w2v
